# Fine-tune Llama 3.1 (QLoRA / PEFT)
This notebook prepares data, runs a small pilot fine-tune using PEFT (LoRA/QLoRA), and contains an MCP checkpoint upload hook.
Adjust `MODEL_PATH` to your local base model, and set `MCP_ENDPOINT` if you use the MCP server.

In [ ]:
# Install requirements (run once)
# !pip install -r requirements.txt

In [ ]:
import os
from pathlib import Path
import pandas as pd
import json

# Set Ollama model path
os.environ['OLLAMA_MODELS'] = 'D:\\ollama_models'

# Paths
ROOT = Path('.')
DATA_CSV = ROOT / 'finance moule' / 'InternationalDeclarations.csv'
FT_DATA_DIR = ROOT / 'ft_data'
FT_DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = ROOT / 'outputs' / 'ft-run'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data CSV: {DATA_CSV}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Ollama models at: {os.environ['OLLAMA_MODELS']}")

# Load CSV and convert to instruction-response pairs
df = pd.read_csv(DATA_CSV)
print(f"Loaded {len(df)} rows from CSV")

# Sample to 25,000 rows for efficient fine-tuning
SAMPLE_SIZE = 25000
if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42)
    print(f"Sampled to {len(df)} rows")

print(f"Columns: {list(df.columns[:5])}... ({len(df.columns)} total)")

# Create instruction-response pairs for finance/permit domain
def create_training_pairs(df):
    """Convert CSV rows to instruction-response pairs for fine-tuning."""
    pairs = []
    for idx, row in df.iterrows():
        # Use concept:name as instruction, and summarize metadata as response
        concept = row.get('concept:name', 'UNKNOWN')
        permit_id = row.get('case:id', row.get('id', ''))
        org_role = row.get('org:role', '')
        amount = row.get('case:Amount', '')
        
        instruction = f"Describe the following financial event: {concept} for permit {permit_id}"
        response = f"Event: {concept}. Role: {org_role}. Amount: {amount}. Status: processed."
        
        pairs.append({
            'instruction': instruction,
            'input': '',
            'output': response
        })
    return pairs

pairs = create_training_pairs(df)
train_jsonl = FT_DATA_DIR / 'train.jsonl'
with open(train_jsonl, 'w', encoding='utf8') as f:
    for pair in pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + '\n')

print(f"Created {len(pairs)} instruction-response pairs")
print(f"Saved to: {train_jsonl}")
print(f"\nExample pair:\n{json.dumps(pairs[0], indent=2)}")

Data CSV: finance moule\InternationalDeclarations.csv
Output directory: outputs\ft-run
Ollama models at: D:\ollama_models
Loaded 72151 rows from CSV
Columns: ['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']... (23 total)
Created 72151 instruction-response pairs
Saved to: ft_data\train.jsonl

Example pair:
{
  "instruction": "Describe the following financial event: Start trip for permit declaration 76457",
  "input": "",
  "output": "Event: Start trip. Role: EMPLOYEE. Amount: 39.66456144659199. Status: processed."
}


## Training (example using QLoRA + PEFT)
The following cell shows a compact training routine. It expects a local base model (4/8-bit compatible with bitsandbytes). Adjust `MODEL_PATH` and training hyperparams for your hardware. This is a minimal, reproducible example — tune for production.

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, 
    TrainingArguments, Trainer, DataCollatorForSeq2Seq,
    BitsAndBytesConfig
)
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import os

# Use Mistral-7B for good performance-to-size ratio
MODEL_PATH = 'mistralai/Mistral-7B-v0.1'

OUTPUT_DIR = 'outputs/ft-run'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading dataset...")
ds = load_dataset('json', data_files=str(FT_DATA_DIR / 'train.jsonl'))
print(f"Dataset size: {len(ds['train'])} examples")

# Build text for training
def make_text(example):
    inst = example.get('instruction', '')
    inp = example.get('input', '')
    out = example.get('output', '')
    if inp:
        text = f"### Instruction:\n{inst}\n\n### Input:\n{inp}\n\n### Response:\n{out}\n"
    else:
        text = f"### Instruction:\n{inst}\n\n### Response:\n{out}\n"
    return {'text': text}

ds = ds.map(make_text, remove_columns=ds['train'].column_names)

print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# === HARDWARE OPTIMIZATION: 4-bit Quantization ===
print("Configuring 4-bit quantization for memory efficiency...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True,
    attn_implementation="flash_attention_2"  # Faster attention if available
)

print("Preparing model for k-bit training...")
model = prepare_model_for_kbit_training(model)

# === HARDWARE OPTIMIZATION: Low-Rank Adaptation (LoRA) ===
print("Attaching LoRA adapters (minimal trainable parameters)...")
lora_config = LoraConfig(
    r=8,  # Reduced from 16 for fewer parameters (~25M trainable)
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # Only attention projections
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Tokenize dataset (shorter sequences = less memory)
def tokenize_fn(batch):
    toks = tokenizer(batch['text'], truncation=True, max_length=256, padding='max_length')
    toks['labels'] = toks['input_ids'].copy()
    return toks

tok_ds = ds.map(tokenize_fn, batched=True)
tok_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

data_collator = DataCollatorForSeq2Seq(
    tokenizer, pad_to_multiple_of=8, return_tensors='pt', padding=True
)

# === HARDWARE OPTIMIZATION: Training Configuration ===
print("Configuring training for resource-constrained hardware...")
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,  # Minimal batch size
    gradient_accumulation_steps=4,  # Simulate batch_size=4 without memory overhead
    gradient_checkpointing=True,  # Trade compute for memory (recompute during backprop)
    num_train_epochs=1,  # 1 epoch is often enough for fine-tuning
    learning_rate=1e-4,  # Conservative learning rate
    fp16=True,  # Half-precision floating point
    optim='paged_adamw_8bit',  # 8-bit optimizer (saves memory)
    save_strategy='epoch',
    logging_steps=50,
    report_to=[],
    save_total_limit=1,  # Keep only 1 checkpoint
    remove_unused_columns=False
)

print(f"Training with {len(tok_ds['train'])} examples...")
print("This will use aggressive memory optimization: 4-bit quantization, LoRA, gradient checkpointing")
print(f"Estimated memory usage: ~3-4 GB VRAM\n")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tok_ds['train'],
    data_collator=data_collator,
)

trainer.train()
print("Training complete!")

# Save final model (only LoRA adapters, ~30 MB)
final_model_path = OUTPUT_DIR + '/final'
trainer.save_model(final_model_path)
print(f"LoRA adapters saved to: {final_model_path}")
print(f"To merge with base model later: model.merge_and_unload()")

Loading dataset...
Dataset size: 72151 examples
Loading tokenizer and model...


d:\Research\Implementation\Fine-tune-Llama\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hansa\.cache\huggingface\hub\models--mistralai--Mistral-7B-v0.1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]d:\Research\Implementation\Fine-tune-Ll

## Next steps / notes
- Confirm `MODEL_PATH` points to your base Llama 3.1 model files.
- If you used a prior technique, replace the model + LoRA config to match it.
- For larger runs, switch to `accelerate launch` and a tuned config.
- I can adapt this notebook to exactly match your previous training recipe if you provide the script/config.